In [ ]:
"""
Standardized plotting module for consistent visualizations.
All functions return matplotlib axes objects for further customization.
"""

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from typing import Optional, List, Union, Tuple, Dict, Any
import pandas as pd


# ============================================================================
# CONFIGURATION & STYLES
# ============================================================================

STYLE_CONFIG = {
    'figure.figsize': (10, 6),
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'font.size': 11,
}

COLOR_PALETTES = {
    'default': sns.color_palette("husl", 8),
    'categorical': sns.color_palette("Set2", 8),
    'sequential': sns.color_palette("viridis", 8),
    'diverging': sns.color_palette("RdYlBu", 8),
}


def set_plot_style(style: str = 'whitegrid', palette: str = 'default'):
    """Set global plotting style."""
    sns.set_style(style)
    sns.set_palette(COLOR_PALETTES.get(palette, COLOR_PALETTES['default']))
    plt.rcParams.update(STYLE_CONFIG)


# ============================================================================
# DISTRIBUTION PLOTS
# ============================================================================

def plot_histogram(
    data: Union[np.ndarray, pd.Series, List],
    ax: Optional[plt.Axes] = None,
    bins: Union[int, str] = 'auto',
    title: Optional[str] = None,
    xlabel: Optional[str] = None,
    ylabel: str = 'Frequency',
    color: Optional[str] = None,
    alpha: float = 0.7,
    kde: bool = True,
    stats_box: bool = True,
    **kwargs
) -> plt.Axes:
    """
    Create a histogram with optional KDE overlay and statistics box.
    
    Args:
        data: 1D array-like data
        ax: Matplotlib axes object (creates new if None)
        bins: Number of bins or method ('auto', 'fd', 'sturges')
        title: Plot title
        xlabel: X-axis label
        ylabel: Y-axis label
        color: Bar color
        alpha: Transparency
        kde: Whether to overlay KDE
        stats_box: Whether to show statistics box
        **kwargs: Additional arguments passed to plt.hist
    
    Returns:
        Matplotlib axes object
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))
    
    # Plot histogram
    n, bins_edges, patches = ax.hist(
        data, bins=bins, alpha=alpha, color=color,
        edgecolor='black', linewidth=0.5, **kwargs
    )
    
    # Add KDE if requested
    if kde:
        from scipy import stats
        kde_x = np.linspace(np.min(data), np.max(data), 200)
        kde = stats.gaussian_kde(data)
        kde_y = kde(kde_x)
        # Scale KDE to match histogram
        kde_y = kde_y * len(data) * (bins_edges[1] - bins_edges[0])
        ax2 = ax.twinx()
        ax2.plot(kde_x, kde_y, color='red', linewidth=2, label='KDE')
        ax2.set_ylabel('Density', color='red')
        ax2.tick_params(axis='y', labelcolor='red')
        ax2.legend(loc='upper right')
    
    # Add statistics box
    if stats_box:
        stats_text = (
            f'n = {len(data)}\n'
            f'μ = {np.mean(data):.3f}\n'
            f'σ = {np.std(data):.3f}\n'
            f'median = {np.median(data):.3f}'
        )
        ax.text(0.02, 0.98, stats_text, transform=ax.transAxes,
                verticalalignment='top', bbox=dict(boxstyle='round',
                facecolor='wheat', alpha=0.5), fontsize=9)
    
    ax.set_xlabel(xlabel or 'Value')
    ax.set_ylabel(ylabel)
    ax.set_title(title or 'Histogram')
    ax.grid(True, alpha=0.3)
    
    return ax


def plot_violin(
    data: Union[pd.DataFrame, Dict[str, np.ndarray]],
    ax: Optional[plt.Axes] = None,
    orient: str = 'v',
    title: Optional[str] = None,
    xlabel: Optional[str] = None,
    ylabel: Optional[str] = None,
    show_points: bool = False,
    **kwargs
) -> plt.Axes:
    """
    Create violin plot for comparing distributions.
    
    Args:
        data: DataFrame or dict of arrays
        ax: Matplotlib axes object
        orient: 'v' (vertical) or 'h' (horizontal)
        title: Plot title
        xlabel/ylabel: Axis labels
        show_points: Whether to overlay individual points
        **kwargs: Passed to sns.violinplot
    
    Returns:
        Matplotlib axes object
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))
    
    # Convert dict to DataFrame if needed
    if isinstance(data, dict):
        data = pd.DataFrame(data)
    
    # Create violin plot
    sns.violinplot(data=data, ax=ax, orient=orient, **kwargs)
    
    # Optionally overlay points
    if show_points:
        sns.stripplot(data=data, ax=ax, orient=orient, 
                     color='black', alpha=0.3, size=3)
    
    ax.set_title(title or 'Violin Plot')
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3, axis='y' if orient == 'v' else 'x')
    
    return ax


def plot_boxplot(
    data: Union[pd.DataFrame, Dict[str, np.ndarray]],
    ax: Optional[plt.Axes] = None,
    orient: str = 'v',
    title: Optional[str] = None,
    xlabel: Optional[str] = None,
    ylabel: Optional[str] = None,
    show_outliers: bool = True,
    notch: bool = False,
    **kwargs
) -> plt.Axes:
    """
    Create box plot for comparing distributions.
    
    Args:
        data: DataFrame or dict of arrays
        ax: Matplotlib axes object
        orient: 'v' (vertical) or 'h' (horizontal)
        title: Plot title
        xlabel/ylabel: Axis labels
        show_outliers: Whether to show outlier points
        notch: Whether to show notched boxes
        **kwargs: Passed to sns.boxplot
    
    Returns:
        Matplotlib axes object
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))
    
    # Convert dict to DataFrame if needed
    if isinstance(data, dict):
        data = pd.DataFrame(data)
    
    # Create box plot
    sns.boxplot(
        data=data, ax=ax, orient=orient,
        showfliers=show_outliers, notch=notch,
        **kwargs
    )
    
    ax.set_title(title or 'Box Plot')
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3, axis='y' if orient == 'v' else 'x')
    
    return ax


# ============================================================================
# TIME SERIES / LINE PLOTS
# ============================================================================

def plot_line(
    x: Optional[np.ndarray] = None,
    y: Union[np.ndarray, List[np.ndarray]] = None,
    ax: Optional[plt.Axes] = None,
    labels: Optional[List[str]] = None,
    title: Optional[str] = None,
    xlabel: Optional[str] = None,
    ylabel: Optional[str] = None,
    colors: Optional[List[str]] = None,
    linestyles: Optional[List[str]] = None,
    markers: Optional[List[str]] = None,
    confidence_intervals: Optional[List[Tuple[np.ndarray, np.ndarray]]] = None,
    show_legend: bool = True,
    **kwargs
) -> plt.Axes:
    """
    Create line plot with optional confidence intervals.
    
    Args:
        x: X-axis values (if None, uses indices)
        y: Y-axis values (single array or list of arrays for multiple lines)
        ax: Matplotlib axes object
        labels: Labels for each line
        title: Plot title
        xlabel/ylabel: Axis labels
        colors: Colors for each line
        linestyles: Line styles for each line
        markers: Markers for each line
        confidence_intervals: List of (lower, upper) bound tuples
        show_legend: Whether to show legend
        **kwargs: Passed to ax.plot
    
    Returns:
        Matplotlib axes object
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 6))
    
    # Handle single vs multiple lines
    if isinstance(y, list):
        y_data = y
    else:
        y_data = [y]
    
    if x is None:
        x = np.arange(len(y_data[0]))
    
    # Plot each line
    for i, y_line in enumerate(y_data):
        label = labels[i] if labels and i < len(labels) else f'Line {i+1}'
        color = colors[i] if colors and i < len(colors) else None
        linestyle = linestyles[i] if linestyles and i < len(linestyles) else '-'
        marker = markers[i] if markers and i < len(markers) else None
        
        ax.plot(x, y_line, label=label, color=color, 
               linestyle=linestyle, marker=marker, **kwargs)
        
        # Add confidence interval if provided
        if confidence_intervals and i < len(confidence_intervals):
            lower, upper = confidence_intervals[i]
            ax.fill_between(x, lower, upper, alpha=0.2, color=color)
    
    ax.set_title(title or 'Line Plot')
    ax.set_xlabel(xlabel or 'X')
    ax.set_ylabel(ylabel or 'Y')
    ax.grid(True, alpha=0.3)
    
    if show_legend and (labels or len(y_data) > 1):
        ax.legend()
    
    return ax


def plot_timeseries(
    data: Union[pd.DataFrame, pd.Series],
    ax: Optional[plt.Axes] = None,
    title: Optional[str] = None,
    ylabel: Optional[str] = None,
    rolling_window: Optional[int] = None,
    show_trend: bool = False,
    **kwargs
) -> plt.Axes:
    """
    Create time series plot with optional rolling average and trend.
    
    Args:
        data: Time series data (DataFrame or Series with datetime index)
        ax: Matplotlib axes object
        title: Plot title
        ylabel: Y-axis label
        rolling_window: Window size for rolling average
        show_trend: Whether to show linear trend
        **kwargs: Passed to plot_line
    
    Returns:
        Matplotlib axes object
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(14, 6))
    
    if isinstance(data, pd.Series):
        data = data.to_frame()
    
    # Plot original data
    for col in data.columns:
        ax.plot(data.index, data[col], label=col, alpha=0.7)
    
    # Add rolling average
    if rolling_window:
        for col in data.columns:
            rolling = data[col].rolling(window=rolling_window).mean()
            ax.plot(data.index, rolling, label=f'{col} (MA{rolling_window})',
                   linewidth=2, linestyle='--')
    
    # Add trend line
    if show_trend:
        from scipy import stats
        for col in data.columns:
            x_numeric = np.arange(len(data))
            slope, intercept, _, _, _ = stats.linregress(x_numeric, data[col])
            trend = slope * x_numeric + intercept
            ax.plot(data.index, trend, label=f'{col} trend',
                   linestyle=':', linewidth=2)
    
    ax.set_title(title or 'Time Series')
    ax.set_xlabel('Time')
    ax.set_ylabel(ylabel or 'Value')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Rotate x-axis labels for better readability
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    return ax


# ============================================================================
# COMPARISON PLOTS
# ============================================================================

def plot_scatter(
    x: np.ndarray,
    y: np.ndarray,
    ax: Optional[plt.Axes] = None,
    title: Optional[str] = None,
    xlabel: Optional[str] = None,
    ylabel: Optional[str] = None,
    color: Optional[Union[str, np.ndarray]] = None,
    size: Optional[Union[float, np.ndarray]] = None,
    show_regression: bool = False,
    show_identity: bool = False,
    colorbar: bool = False,
    **kwargs
) -> plt.Axes:
    """
    Create scatter plot with optional regression line.
    
    Args:
        x, y: Data arrays
        ax: Matplotlib axes object
        title: Plot title
        xlabel/ylabel: Axis labels
        color: Color or array for color mapping
        size: Point size or array for size mapping
        show_regression: Whether to show regression line
        show_identity: Whether to show y=x line
        colorbar: Whether to show colorbar (if color is array)
        **kwargs: Passed to ax.scatter
    
    Returns:
        Matplotlib axes object
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8))
    
    # Create scatter plot
    scatter = ax.scatter(x, y, c=color, s=size, alpha=0.6, **kwargs)
    
    # Add colorbar if needed
    if colorbar and isinstance(color, np.ndarray):
        plt.colorbar(scatter, ax=ax)
    
    # Add regression line
    if show_regression:
        from scipy import stats
        slope, intercept, r_value, _, _ = stats.linregress(x, y)
        line_x = np.array([x.min(), x.max()])
        line_y = slope * line_x + intercept
        ax.plot(line_x, line_y, 'r--', linewidth=2,
               label=f'y = {slope:.3f}x + {intercept:.3f}\nR² = {r_value**2:.3f}')
        ax.legend()
    
    # Add identity line
    if show_identity:
        lims = [
            np.min([ax.get_xlim(), ax.get_ylim()]),
            np.max([ax.get_xlim(), ax.get_ylim()]),
        ]
        ax.plot(lims, lims, 'k--', alpha=0.5, zorder=0, label='y=x')
        ax.set_xlim(lims)
        ax.set_ylim(lims)
    
    ax.set_title(title or 'Scatter Plot')
    ax.set_xlabel(xlabel or 'X')
    ax.set_ylabel(ylabel or 'Y')
    ax.grid(True, alpha=0.3)
    
    return ax


def plot_heatmap(
    data: Union[np.ndarray, pd.DataFrame],
    ax: Optional[plt.Axes] = None,
    title: Optional[str] = None,
    cmap: str = 'viridis',
    annot: bool = False,
    fmt: str = '.2f',
    cbar: bool = True,
    **kwargs
) -> plt.Axes:
    """
    Create heatmap visualization.
    
    Args:
        data: 2D array or DataFrame
        ax: Matplotlib axes object
        title: Plot title
        cmap: Colormap name
        annot: Whether to annotate cells with values
        fmt: Format string for annotations
        cbar: Whether to show colorbar
        **kwargs: Passed to sns.heatmap
    
    Returns:
        Matplotlib axes object
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8))
    
    sns.heatmap(data, ax=ax, cmap=cmap, annot=annot, fmt=fmt,
                cbar=cbar, **kwargs)
    
    ax.set_title(title or 'Heatmap')
    
    return ax


# ============================================================================
# MULTI-PANEL PLOTS
# ============================================================================

def create_comparison_grid(
    plot_funcs: List[callable],
    plot_data: List[Dict[str, Any]],
    nrows: Optional[int] = None,
    ncols: Optional[int] = None,
    figsize: Optional[Tuple[int, int]] = None,
    suptitle: Optional[str] = None
) -> Tuple[plt.Figure, np.ndarray]:
    """
    Create grid of subplots for comparison.
    
    Args:
        plot_funcs: List of plotting functions
        plot_data: List of dicts containing kwargs for each plot
        nrows/ncols: Grid dimensions (auto-calculated if None)
        figsize: Figure size
        suptitle: Overall title
    
    Returns:
        Figure and array of axes
    """
    n_plots = len(plot_funcs)
    
    if nrows is None and ncols is None:
        ncols = int(np.ceil(np.sqrt(n_plots)))
        nrows = int(np.ceil(n_plots / ncols))
    elif nrows is None:
        nrows = int(np.ceil(n_plots / ncols))
    elif ncols is None:
        ncols = int(np.ceil(n_plots / nrows))
    
    if figsize is None:
        figsize = (6 * ncols, 5 * nrows)
    
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.atleast_1d(axes).flatten()
    
    for i, (func, data) in enumerate(zip(plot_funcs, plot_data)):
        func(ax=axes[i], **data)
    
    # Hide unused axes
    for i in range(n_plots, len(axes)):
        axes[i].set_visible(False)
    
    if suptitle:
        fig.suptitle(suptitle, fontsize=16, y=0.995)
    
    plt.tight_layout()
    
    return fig, axes


# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def save_plot(fig: plt.Figure, filename: str, dpi: int = 300, **kwargs):
    """Save figure to file."""
    fig.savefig(filename, dpi=dpi, bbox_inches='tight', **kwargs)
    print(f"Plot saved to {filename}")


def show_plot():
    """Display all open plots."""
    plt.show()


# ============================================================================
# EXAMPLE USAGE
# ============================================================================

if __name__ == "__main__":
    # Set style
    set_plot_style()
    
    # Generate sample data
    np.random.seed(42)
    data = np.random.randn(1000)
    
    # Example 1: Histogram
    fig1, ax1 = plt.subplots()
    plot_histogram(data, ax=ax1, title="Distribution Example", 
                  xlabel="Value", kde=True, stats_box=True)
    
    # Example 2: Multiple violins
    violin_data = {
        'Group A': np.random.randn(100),
        'Group B': np.random.randn(100) + 1,
        'Group C': np.random.randn(100) - 0.5
    }
    fig2, ax2 = plt.subplots()
    plot_violin(violin_data, ax=ax2, title="Group Comparison")
    
    # Example 3: Time series
    dates = pd.date_range('2023-01-01', periods=100)
    ts_data = pd.Series(np.cumsum(np.random.randn(100)), index=dates)
    fig3, ax3 = plt.subplots()
    plot_timeseries(ts_data, ax=ax3, rolling_window=10, show_trend=True)
    
    plt.show()